In [2]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("/Users/riverocel/Downloads/marketing_and_product_performance.csv")
df.head()
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Campaign_ID                        10000 non-null  object 
 1   Product_ID                         10000 non-null  object 
 2   Budget                             10000 non-null  float64
 3   Clicks                             10000 non-null  int64  
 4   Conversions                        10000 non-null  int64  
 5   Revenue_Generated                  10000 non-null  float64
 6   ROI                                10000 non-null  float64
 7   Customer_ID                        10000 non-null  object 
 8   Subscription_Tier                  10000 non-null  object 
 9   Subscription_Length                10000 non-null  int64  
 10  Flash_Sale_ID                      10000 non-null  object 
 11  Discount_Level                     10000 non-null  int6

In [22]:
target = "ROI"

X = df.drop(columns=[
    target,
    "Campaign_ID",
    "Product_ID",
    "Customer_ID",
    "Flash_Sale_ID",
    "Bundle_ID"
])

y = df[target]
categorical = X.select_dtypes(include="object").columns.tolist()
numeric = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical variables:")
print(categorical)

print("\nNumeric variables:")
print(numeric)

Categorical variables:
['Subscription_Tier', 'Common_Keywords']

Numeric variables:
['Budget', 'Clicks', 'Conversions', 'Revenue_Generated', 'Subscription_Length', 'Discount_Level', 'Units_Sold', 'Bundle_Price', 'Customer_Satisfaction_Post_Refund']


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical)
    ]
)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_test_processed = X_test_processed.toarray()

X_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

print(X_train_df.shape)
X_train_df.head()

(8000, 14)


,num__Budget,num__Clicks,num__Conversions,num__Revenue_Generated,num__Subscription_Length,num__Discount_Level,num__Units_Sold,num__Bundle_Price,num__Customer_Satisfaction_Post_Refund,cat__Subscription_Tier_Premium,cat__Subscription_Tier_Standard,cat__Common_Keywords_Durable,cat__Common_Keywords_Innovative,cat__Common_Keywords_Stylish
0,-1.264876,0.170499,-0.436694,0.826389,0.089523,1.661845,0.182056,-0.220254,-0.442737,1.0,0.0,0.0,1.0,0.0
1,-0.534454,-1.731882,-0.405557,-1.277233,-0.502607,0.209405,-1.595821,-0.942892,0.454854,0.0,0.0,0.0,0.0,1.0
2,-1.470049,-1.239320,-0.710006,0.214018,-0.798672,-0.371570,0.286637,-0.203269,-1.340327,0.0,0.0,0.0,1.0,0.0
3,0.953819,1.070264,1.390003,1.251505,1.372470,-0.429668,-1.282078,0.456061,-0.442737,0.0,0.0,0.0,1.0,0.0
4,0.507874,-1.236522,-1.702927,1.392786,-1.094737,1.603748,-1.264648,1.022591,0.454854,0.0,0.0,0.0,0.0,1.0


# forward selection to pick the best features 

In [27]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

# LR
lr = LinearRegression()

#FS
forward = SequentialFeatureSelector(
    lr,
    n_features_to_select="auto",
    direction="forward",
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

forward.fit(X_train_df, y_train)

selected_features = X_train_df.columns[forward.get_support()]

print("Selected Features:")
print(selected_features.tolist())

Selected Features:
['num__Budget', 'num__Clicks', 'num__Revenue_Generated', 'num__Discount_Level', 'num__Bundle_Price', 'cat__Subscription_Tier_Standard', 'cat__Common_Keywords_Innovative']


best features selected, getting r2 next

In [28]:
forward_model = LinearRegression()

forward_model.fit(
    X_train_df[selected_features],
    y_train
)

pred = forward_model.predict(
    X_test_df[selected_features]
)

rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")

RMSE: 1.302
R²: -0.001


the model actually performed horrible, because of the synthetic data the variables have a really bad connection with ROI, there are essentially not a single strong one.